# PSO pour la sélection d'attributs (classification supervisée)

Optimisation par essaim de particules (PSO) binaire + SVM, testée sur 5 jeux de données :
Arrhythmia, Colon Cancer, Dermatology, Heart Stat, Vehicule.

**Pour lancer sur Google Colab :** mets les 5 CSV dans un dossier `data` à la racine de ton Drive, puis exécute les cellules dans l'ordre.
Pour un test rapide, mets `QUICK_TEST = True` dans la cellule de configuration (quelques minutes au lieu de plusieurs heures).

## 1. Configuration et données

In [1]:
from google.colab import drive
drive.mount("/content/drive")

# Dossier "data" créé à la racine de ton Drive (Mon Drive/data) avec les 5 CSV dedans
DATA_DIR = "/content/drive/MyDrive/data"

# True = test rapide (2 exécutions x 20 itérations), False = expérience complète (10 x 100)
QUICK_TEST = False

N_RUNS   = 2  if QUICK_TEST else 10
MAX_ITER = 20 if QUICK_TEST else 100

Mounted at /content/drive


In [2]:
import os
print("Fichiers trouvés :", sorted(os.listdir(DATA_DIR)))

Fichiers trouvés : ['arrhythmia.csv', 'colon_cancer.csv', 'darmatology.csv', 'heart_stat.csv', 'vehicule.csv']


## 2. Imports

In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

## 3. Algorithme PSO

- Chaque particule est un vecteur de positions réelles ; une conversion **sigmoïde S4 + tirage probabiliste** le transforme en masque binaire (1 = attribut gardé).
- Fitness = `alpha * accuracy - beta * (nb_attributs_choisis / nb_attributs_total)`

In [4]:
class PSO_FeatureSelection:

    def __init__(self, n_particles=30, max_iter=100, w=0.7, c1=1.5, c2=1.5,
                 classifier_type="SVM", alpha=0.99, beta=0.01):
        self.n_particles = n_particles
        self.max_iter = max_iter
        self.w = w                  # inertie
        self.c1 = c1                # coefficient cognitif (mémoire personnelle)
        self.c2 = c2                # coefficient social (influence du groupe)
        self.alpha = alpha          # poids de l'accuracy
        self.beta = beta            # poids du nombre d'attributs
        self.classifier_type = classifier_type
        self.convergence_curve = []

    @staticmethod
    def S4(x):
        """Sigmoïde de paramètre 3."""
        return 1 / (1 + np.exp(-x / 3))

    def _to_binary(self, position):
        """Réel -> binaire : l'attribut i est gardé avec la probabilité S4(position[i])."""
        return (np.random.rand(len(position)) < self.S4(position)).astype(int)

    def _make_classifier(self):
        if self.classifier_type == "SVM":
            return SVC(kernel="rbf", random_state=42)
        if self.classifier_type == "RF":
            return RandomForestClassifier(n_estimators=100, random_state=42)
        return KNeighborsClassifier(n_neighbors=5)

    def _fitness(self, position, X_train, X_test, y_train, y_test):
        mask = self._to_binary(position).astype(bool)
        if not mask.any():                      # au moins 1 attribut
            return 0.0
        clf = self._make_classifier()
        clf.fit(X_train[:, mask], y_train)
        acc = accuracy_score(y_test, clf.predict(X_test[:, mask]))
        return self.alpha * acc - self.beta * (mask.sum() / len(position))

    def optimize(self, X_train, X_test, y_train, y_test):
        n_feat = X_train.shape[1]
        pos = np.random.rand(self.n_particles, n_feat)
        vel = np.random.uniform(-1, 1, (self.n_particles, n_feat))

        pbest_pos = pos.copy()
        pbest_fit = np.array([self._fitness(p, X_train, X_test, y_train, y_test) for p in pos])
        g = pbest_fit.argmax()
        gbest_pos, gbest_fit = pos[g].copy(), pbest_fit[g]

        for it in range(self.max_iter):
            for i in range(self.n_particles):
                r1, r2 = np.random.rand(n_feat), np.random.rand(n_feat)
                vel[i] = (self.w * vel[i]
                          + self.c1 * r1 * (pbest_pos[i] - pos[i])
                          + self.c2 * r2 * (gbest_pos - pos[i]))
                vel[i] = np.clip(vel[i], -1, 1)
                pos[i] = 1 / (1 + np.exp(-(pos[i] + vel[i])))   # position ramenée dans [0, 1]

                fit = self._fitness(pos[i], X_train, X_test, y_train, y_test)
                if fit > pbest_fit[i]:
                    pbest_fit[i], pbest_pos[i] = fit, pos[i].copy()
                    if fit > gbest_fit:
                        gbest_fit, gbest_pos = fit, pos[i].copy()

            self.convergence_curve.append(gbest_fit)
            if (it + 1) % 20 == 0:
                print(f"  Itération {it+1}/{self.max_iter} - meilleure fitness : {gbest_fit:.4f}")

        return self._to_binary(gbest_pos), gbest_fit

## 4. Fonctions utilitaires

In [5]:
def load_dataset(path):
    """Charge un CSV (dernière colonne = classe), encode les labels texte, impute les NaN."""
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values

    if y.dtype == object:
        y = LabelEncoder().fit_transform(y)

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy="mean").fit_transform(X)

    print(f"  {os.path.basename(path)} : {X.shape[0]} échantillons, "
          f"{X.shape[1]} attributs, {len(np.unique(y))} classes")
    return X, y


def evaluate_model(X, y, mask, classifier_type="SVM"):
    """Accuracy, F1, AUC avec uniquement les attributs sélectionnés."""
    X_sel = X[:, mask.astype(bool)]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sel, y, test_size=0.3, random_state=42, stratify=y)

    if classifier_type == "SVM":
        clf = SVC(kernel="rbf", probability=True, random_state=42)
    elif classifier_type == "RF":
        clf = RandomForestClassifier(n_estimators=100, random_state=42)
    else:
        clf = KNeighborsClassifier(n_neighbors=5)

    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average="weighted")

    try:
        proba = clf.predict_proba(X_te)
        if len(np.unique(y)) == 2:
            auc = roc_auc_score(y_te, proba[:, 1])
        else:
            auc = roc_auc_score(y_te, proba, multi_class="ovr", average="weighted")
    except Exception:
        auc = None

    return acc, f1, auc, int(mask.sum())


def run_experiments(datasets, n_runs=10, max_iter=100, classifier_type="SVM"):
    """Lance le PSO n_runs fois par dataset et renvoie les valeurs de chaque exécution."""
    results = {}
    for name, (X, y) in datasets.items():
        print(f"\n{'='*70}\nDataset : {name}  ({X.shape[0]} x {X.shape[1]})\n{'='*70}")

        X_scaled = StandardScaler().fit_transform(X)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X_scaled, y, test_size=0.3, random_state=42, stratify=y)

        accs, f1s, aucs, nfeats = [], [], [], []
        for run in range(n_runs):
            print(f"--- Exécution {run+1}/{n_runs} ---")
            pso = PSO_FeatureSelection(max_iter=max_iter, classifier_type=classifier_type)
            mask, _ = pso.optimize(X_tr, X_te, y_tr, y_te)

            acc, f1, auc, n = evaluate_model(X_scaled, y, mask, classifier_type)
            accs.append(acc); f1s.append(f1); nfeats.append(n)
            if auc is not None:
                aucs.append(auc)
            print(f"  -> {n}/{X.shape[1]} attributs | accuracy {acc:.4f} | F1 {f1:.4f}")

        results[name] = {"acc": accs, "f1": f1s, "auc": aucs, "nfeat": nfeats}
    return results

## 5. Chargement des datasets

In [6]:
dataset_files = {
    "Arrhythmia":   "arrhythmia.csv",
    "Colon Cancer": "colon_cancer.csv",
    "Dermatology":  "darmatology.csv",
    "Heart Stat":   "heart_stat.csv",
    "Vehicule":     "vehicule.csv",
}

datasets = {name: load_dataset(os.path.join(DATA_DIR, f)) for name, f in dataset_files.items()}

  arrhythmia.csv : 452 échantillons, 279 attributs, 13 classes
  colon_cancer.csv : 62 échantillons, 2000 attributs, 2 classes
  darmatology.csv : 366 échantillons, 34 attributs, 6 classes
  heart_stat.csv : 270 échantillons, 13 attributs, 2 classes
  vehicule.csv : 846 échantillons, 18 attributs, 4 classes


## 6. Expériences

Config : SVM (RBF), 30 particules, `w=0.7`, `c1=c2=1.5`, fitness `α=0.99` / `β=0.01`.
La version complète (10 exécutions x 100 itérations x 5 datasets) est longue, surtout sur Colon Cancer (2000 attributs).

In [7]:
results = run_experiments(datasets, n_runs=N_RUNS, max_iter=MAX_ITER, classifier_type="SVM")


Dataset : Arrhythmia  (452 x 279)
--- Exécution 1/10 ---
  Itération 20/100 - meilleure fitness : 0.6352
  Itération 40/100 - meilleure fitness : 0.6355
  Itération 60/100 - meilleure fitness : 0.6355
  Itération 80/100 - meilleure fitness : 0.6357
  Itération 100/100 - meilleure fitness : 0.6500
  -> 157/279 attributs | accuracy 0.6103 | F1 0.5048
--- Exécution 2/10 ---
  Itération 20/100 - meilleure fitness : 0.6422
  Itération 40/100 - meilleure fitness : 0.6422
  Itération 60/100 - meilleure fitness : 0.6425
  Itération 80/100 - meilleure fitness : 0.6498
  Itération 100/100 - meilleure fitness : 0.6498
  -> 143/279 attributs | accuracy 0.6029 | F1 0.4914
--- Exécution 3/10 ---
  Itération 20/100 - meilleure fitness : 0.6498
  Itération 40/100 - meilleure fitness : 0.6498
  Itération 60/100 - meilleure fitness : 0.6498
  Itération 80/100 - meilleure fitness : 0.6498
  Itération 100/100 - meilleure fitness : 0.6498
  -> 159/279 attributs | accuracy 0.6103 | F1 0.5005
--- Exécution 

## 7. Tableau récapitulatif et sauvegarde

In [8]:
rows = []
for name, r in results.items():
    rows.append({
        "Dataset": name,
        "Moyenne Précision": np.mean(r["acc"]),
        "Écart-type Précision": np.std(r["acc"]),
        "Min Précision": np.min(r["acc"]),
        "Max Précision": np.max(r["acc"]),
        "Médiane Précision": np.median(r["acc"]),
        "Moy Nb Attributs": np.mean(r["nfeat"]),
        "Écart-type Attributs": np.std(r["nfeat"]),
    })

df_results = pd.DataFrame(rows).round(4)
display(df_results)

out_path = os.path.join(DATA_DIR, "resultats_pso.csv")
df_results.to_csv(out_path, index=False)
print("Résultats sauvegardés dans :", out_path)

,Dataset,Moyenne Précision,Écart-type Précision,Min Précision,Max Précision,Médiane Précision,Moy Nb Attributs,Écart-type Attributs
0,Arrhythmia,0.6015,0.0131,0.5735,0.6103,0.6103,156.4,8.1388
1,Colon Cancer,0.7895,0.0000,0.7895,0.7895,0.7895,1103.5,26.8263
2,Dermatology,0.9336,0.0312,0.8636,0.9545,0.9455,19.5,3.2939
3,Heart Stat,0.7889,0.0470,0.7037,0.8765,0.7778,6.9,1.8138
4,Vehicule,0.7185,0.0207,0.6811,0.7480,0.7185,10.6,1.4283


Résultats sauvegardés dans : /content/drive/MyDrive/data/resultats_pso.csv
